In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
# ROLL: Data Loading (Andmete laadimine) -- Krista

# sales andmete laadimine.
df_sales = pd.read_csv('sales.csv')
# customers andmete laadimine.
df_customers = pd.read_csv('customers.csv')

# Tabelite ühendamine
df_merged = pd.merge(df_sales, df_customers, on='customer_id', how='left')                    ## Ühendame müügi- ja klienditabelid customer_id

# Kontrollime, et kõik vajalikud veerud on olemas
required_cols = ['customer_id', 'sale_date', 'total_price', 'email']
print("Nõutud veerud olemas:", all(c in df_merged.columns for c in required_cols))

# Kontrollime ühendatud tabeli struktuuri
print("\n--- TABELITE STRUKTUUR ---")
# Kuvab andmete ridade ja veergude arvu
print(df_merged.shape)
# kuvab esimesed 5 elementi tabelist
print(df_merged.head())
# kuvab tabelite andmetüüpe
print(df_merged.dtypes)

In [ ]:
# ROLL: Data Cleaning (Andmete puhastamine) -- Egle

# Kuvame algset ridade arvu
print("Algne ridade arv:", df_merged.shape)

# Duplikaatide kontroll ja eemaldamine
print("Duplikaatide arv:", df_merged.duplicated().sum())
df_cleaned = df_merged.drop_duplicates()

# NULL väärtuste kontroll ja kriitiliste ridade eemaldamine
print("NULL väärtused veergudes:\n", df_cleaned.isnull().sum())

# Eemaldame read, kus puudub ID, kuupäev või hind
df_cleaned = df_cleaned.dropna(subset=['customer_id', 'sale_date', 'total_price'])

# Kuupäevade muutmine õigesse formaati
df_cleaned['sale_date'] = pd.to_datetime(df_cleaned['sale_date'])

# Eemaldame müügid, mis on hilisemad kui viitekuupäev
df_cleaned = df_cleaned[df_cleaned['sale_date'] <= pd.to_datetime('2025-02-28')]

# Vigaste väärtuste eemaldamine
# Jätame alles ainult need read, kus hind on suurem kui 0
df_cleaned = df_cleaned[df_cleaned['total_price'] > 0]

# Kuvame puhastusraporti, et veenduda andmestiku korrektsuses
print("\n--- PUHASTUSRAPORT ---")
print("Lõplik ridade arv (shape):", df_cleaned.shape)
print("Unikaalseid kliente:", df_cleaned['customer_id'].nunique())
print("Periood:", df_cleaned['sale_date'].min(), "kuni", df_cleaned['sale_date'].max())

# Kuvame puhastatud andmete alguse
print("\nValmis andmed analüüsiks:")
print(df_cleaned.head())

In [ ]:
# ROLL: Analysis — RFM kliendisegmenteerimine -- Kevin

# Viitekuupäev on fikseeritud, et tulemused oleksid võrreldavad
today = pd.to_datetime('2025-02-28')

# Samm 2: Arvuta Recency — viimane ostu kuupäev ja päevade arv
recency_df = df_cleaned.groupby('customer_id')['sale_date'].max().reset_index()
recency_df.columns = ['customer_id', 'last_purchase']
recency_df['recency_days'] = (today - recency_df['last_purchase']).dt.days

# Samm 3: Arvuta Frequency — ostude arv
frequency_df = df_cleaned.groupby('customer_id')['sale_id'].count().reset_index()
frequency_df.columns = ['customer_id', 'frequency']

# Samm 4: Arvuta Monetary — kogukulutus
monetary_df = df_cleaned.groupby('customer_id')['total_price'].sum().reset_index()
monetary_df.columns = ['customer_id', 'monetary']

# Samm 5: Liida R, F, M ühte tabelisse pd.merge abil
rfm = pd.merge(recency_df[['customer_id', 'recency_days']], frequency_df, on='customer_id')
rfm = pd.merge(rfm, monetary_df, on='customer_id')

# Määra skoorid 1-5 (pd.qcut jagab andmed 5 võrdsesse gruppi)
# Recency puhul on väiksem number parem, seega sildid on tagurpidi [5, 4, 3, 2, 1]
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1], duplicates='drop')
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5])

# Samm 6: Arvuta summaarne RFM skoor
rfm['RFM_Score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

# Segmenteerimisfunktsioon skooride põhjal
def määra_segment(score):
    if score >= 13: return 'VIP Champions'
    elif score >= 10: return 'Loyal'
    elif score >= 7: return 'Potential'
    elif score >= 4: return 'At Risk'
    else: return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(määra_segment)

# VÄLJUND: RFM kokkuvõttetabel
print("--- RFM SEGMENTIDE KOKKUVÕTE ---")
segmentide_arv = rfm['Segment'].value_counts()
segmentide_protsent = rfm['Segment'].value_counts(normalize=True) * 100

kokkuvotte_df = pd.DataFrame({
    'Klientide arv': segmentide_arv,
    'Osakaal (%)': segmentide_protsent.round(1)
})

print(kokkuvotte_df)

# Kontroll: kas kõik kliendid said segmendi?
print("\nSegmendita kliente:", rfm['Segment'].isnull().sum())

# Kuvame tabeli alguse, et näha tulemusi
print("\nNäidis kliendiandmetest koos skooridega:")
rfm.head()

In [ ]:
# ROLL: Visualization (Visualiseerimine ja leiud) -- Eike
import plotly.express as px
import pandas as pd

# ==========================================
# TURVAKONTROLL ENNE JOONISTAMIST (Garanteerib tööle hakkamise)
# ==========================================
rfm_plot = rfm.copy()

# 1. Kui veergudel on kahekordne päis (MultiIndex), teeme selle lamedaks
if isinstance(rfm_plot.columns, pd.MultiIndex):
    rfm_plot.columns = [
        col[1] if col[1] else col[0] for col in rfm_plot.columns
    ]

# 2. Toome customer_id indeksist tavaliseks veeruks
rfm_plot = rfm_plot.reset_index()

# 3. Kaitseriiv: veendume, et veergude nimed on väikesed (Plotly jaoks)
# Kui su veergude nimed on nt 'Recency' või 'Monetary', tee need siin samaks, mis Plotlys
rfm_plot.columns = rfm_plot.columns.str.lower()

# 4. Plotly 'size' ei salli nulle ega miinuseid
rfm_plot = rfm_plot[rfm_plot["frequency"] > 0].dropna()
# ==========================================


# DIAGRAMM 1: Segmentide jaotus (Tulpdiagramm)
segment_counts = rfm_plot["segment"].value_counts().reset_index()
segment_counts.columns = ["Segment", "Klientide arv"]

fig1 = px.bar(
    segment_counts,
    x="Segment",
    y="Klientide arv",
    title="Kliendibaasi jaotus segmentide lõikes",
    color="Segment",
    text_auto=True,
)
fig1.show()


# DIAGRAMM 2: Klientide käitumine (Hajuvusdiagramm)
# DIAGRAMM 2: Klientide käitumine (Hajuvusdiagramm)
fig2 = px.scatter(
    rfm_plot,  # Using the cleaned rfm_plot table
    x="recency_days",  # Fixed: changed "recency" to "recency_days"
    y="monetary",
    size="frequency",
    color="segment",
    hover_name="customer_id",
    title="Klientide väärtus vs. värskus (Mulli suurus = ostude arv)",
    labels={
        "recency_days": "Päevi viimasest ostust",  # Maps the clean label to the correct column
        "monetary": "Kogukulu (€)",
        "segment": "Segment",
    },
)
fig2.show()

# DIAGRAMM 3: Top 10 VIP klienti
# Otsime VIP-e (kontrolli, et su andmetes kirjutatakse see täpselt nii, vajadusel väiketähtedega)
top_vips = rfm_plot[rfm_plot["segment"].str.lower() == "vip champions"].nlargest(
    10, "monetary"
)

fig3 = px.bar(
    top_vips,
    x="customer_id",  # Kuna tegime reset_index(), on customer_id nüüd puhas veerg!
    y="monetary",
    title="Top 10 VIP klienti kogukulutuse järgi",
    labels={"customer_id": "Kliendi ID", "monetary": "Kogukulu (€)"},
    color_discrete_sequence=["#FFD700"],
)  # Kuldne värv VIP-dele
fig3.show()


# --- ÄRITÕLGENDUS MARKOLLE ---
print("\n--- KOKKUVÕTE MARKOLLE ---")
vip_count = len(rfm_plot[rfm_plot["segment"].str.lower() == "vip champions"])
at_risk_count = len(rfm_plot[rfm_plot["segment"].str.lower() == "at risk"])
total_revenue = rfm_plot["monetary"].sum()
vip_revenue = rfm_plot[rfm_plot["segment"].str.lower() == "vip champions"][
    "monetary"
].sum()

# Kui VIP-e leiti, arvutame osakaalu, muidu kuvame 0
vip_share = (vip_revenue / total_revenue) * 100 if total_revenue > 0 else 0

print(
    f"Meie andmetes on {vip_count} VIP-klienti, kes moodustavad tervelt {vip_share:.1f}% kogu käibest."
)
print(
    f"Samas on meil {at_risk_count} klienti staatuses 'At Risk', kes pole ammu ostnud."
)
print(
    "Peamine fookus peaks olema VIP-ide hoidmisel ja ootel olevate klientide reaktiveerimisel."
)

# --- KONKREETSED SOOVITUSED ---
print("\n--- SOOVITUSED ---")
print(
    "1. VIP PROGRAMM: Pakkuda top 10 kliendile personaalset teenindust ja varajast ligipääsu uutele toodetele."
)
print(
    "2. WIN-BACK KAMPAANIA: Saata 'At Risk' segmendile sooduskood, et motiveerida neid uut ostu tegema."
)
print(
    "3. NURTURE PROGRAMM: 'Potential' segmendile suunata sisuturundust, et tõsta nende ostusagedust."
)